在深度学习的发展历程中，研究人员从**绝对位置、相对位置、复数空间、连续隐空间**等多个维度探索了如何为模型注入顺序信息。

以下是深度学习领域中几乎所有主流及重要边缘化的位置编码（Positional Encoding/Embedding）方案的系统性汇总：

---

### 一、 绝对位置编码 (Absolute Positional Encodings)

这类方法直接为每个绝对坐标（如第 $t$ 个 Token）分配一个独立的向量。

1. **Sinusoidal Positional Encoding (正弦/余弦固定编码)**
* **来源**：Transformer 奠基论文 *Attention Is All You Need* (2017)。
* **原理**：使用不同频率的 $\sin$ 和 $\cos$ 函数组合生成固定的高维向量。


2. **Learned Absolute Position Embedding (可学习绝对位置嵌入)**
* **来源**：BERT, GPT 系列 (2018)。
* **原理**：将位置视作离散 ID（0, 1, 2...），初始化一个类似词表的可学习矩阵 `[Max_Seq_Len, d_model]`，通过反向传播训练。


3. **Floater (连续绝对位置编码)**
* **来源**：由刘一佳等提出 (2020)。
* **原理**：用神经网络（如常微分方程 ODE 或递归网络）将绝对位置建模为连续空间中的轨迹，允许对未见过的绝对位置进行内插和外推。



---

### 二、 相对位置编码 (Relative Positional Encodings)

这类方法不关心具体的绝对坐标，而是显式或隐式地对 Token 之间的相对距离（$i - j$）进行建模。

4. **Shaw's Relative Position Embedding (Shaw 相对位置)**
* **来源**：*Self-Attention with Relative Position Representations* (2018)。
* **原理**：在计算 Attention 的 $Q K^T$ 和值向量 $V$ 时，分别加入可学习的相对位置截断向量 $a_{ij}^K$ 和 $a_{ij}^V$。限制最大相对距离为 $k$（超过 $k$ 的距离共享同一个编码）。


5. **Transformer-XL PE (截断式相对位置)**
* **来源**：Transformer-XL (2019)。
* **原理**：为了支持跨片段（Segment）的循环机制，将 $Q K^T$ 展开，把绝对位置项替换为正弦编码的相对距离向量 $R_{i-j}$，并引入了两个全局偏置向量 $u$ 和 $v$。


6. **T5 Relative Bias (T5 相对位置偏置)**
* **来源**：Google T5 模型 (2019)。
* **原理**：最简化的相对位置设计。不在 Embedding 层做任何操作，而是直接在计算好的 Attention 分数矩阵 $A_{ij}$ 上加上一个标量偏置 $b_{i-j}$。偏置通过分桶（Bucket）机制实现非线性缩放（距离越近分辨越精细，距离越远桶越粗）。


7. **DeBERTa Relative PE (解耦注意力相对位置)**
* **来源**：微软 DeBERTa (2020)。
* **原理**：将内容（Content）和位置（Position）完全解耦。计算 Attention 分数时拆分为四项：内容-内容、内容-位置、位置-内容、位置-位置，均采用相对位置矩阵。


8. **KERPLE (Kernelized Relative Position Embedding)**
* **来源**：*Kernelized Relative Positional Embedding for Long Sequences* (2022)。
* **原理**：用核函数（如高斯核、拉普拉斯核）来数学化地显式建模相对位置的衰减趋势。



---

### 三、 混合与跨空间位置编码 (Hybrid & Mathematical Space PEs)

利用复数空间、矩阵旋转或几何变换，兼顾绝对位置的实现便利性与相对位置的数学优良性。

9. **RoPE (Rotary Position Embedding, 旋转位置编码)**
* **来源**：苏剑林 *Roformer* (2021) / LLaMA 广泛采用。
* **原理**：将二维子空间中的向量乘以一个正交旋转矩阵，旋转角度与绝对位置成正比。在内积操作下天然转化为相对位置。


10. **Complex-linear PE (复数线性位置编码)**
* **来源**：ICLR 2020 *Encoding word order in complex embeddings*。
* **原理**：将普通词嵌入拓展到复数空间，每个词用复数表示，而位置信息则作为复数的相位角（Phase Angle）叠加到词向量上。


11. **XPos (Exponential Distance Decay RoPE)**
* **来源**：微软 *A Length-Extrapolatable Transformer* (2022)，常用于 Meta-Transformer。
* **原理**：在 RoPE 的基础上引入了一个指数衰减阻尼因子，使得模型在旋转的基础上，随着相对距离增加，注意力权重呈现平滑的指数级衰减，大幅增强了长文本外推性。



---

### 四、 现代大模型长文本外推/插值位置编码 (Extrapolation & Interpolation PEs)

当模型需要处理远超训练长度（例如从 4k 扩展到 128k 甚至 1M）的文本时，专门针对 RoPE 进行修正的变体方案：

12. **Linear Interpolation / Position Interpolation (PI, 位置插值)**
* **来源**：Meta (2023)。
* **原理**：直接将超出训练长度的位置等比例“压缩”（插值）到原有的训练窗口内（如位置 $pos \to pos / N$）。


13. **NTK-aware Scaled RoPE**
* **来源**：开源社区开源贡献者 / 随后被各个大模型采纳。
* **原理**：基于神经切线核（NTK）理论，对高频和低频维度进行非均匀缩放。高频部分（微观位置）不插值保持精度，低频部分（宏观位置）进行插值，避免了低频信息的模糊。


14. **Dynamic NTK / YaRN (Yet another RoPE extensioN)**
* **来源**：YaRN 团队 (2023)。
* **原理**：目前最先进的 RoPE 扩展方案之一。在 NTK 基础上加入注意力矩阵的修正因子（Attention Spreading Correction），解决插值后注意力分布变平淡、模型困惑度（PPL）上升的问题。


15. **SuMMER (Symmetric Multi-Resolution RoPE)**
* **来源**：针对超长上下文的变体。
* **原理**：利用多分辨率机制分配不同的频域旋转速度，专门适配百万级（1M+）上下文。



---

### 五、 无参数/隐式位置编码 (Parameter-free & Implicit PEs)

16. **No Positional Encoding (无位置编码 / NoPE)**
* **来源**：部分学者研究及特定架构（如带有特定 Padding 或 Casual Mask 的 Decoder-Only 模型）。
* **原理**：完全不加位置编码。某些研究表明，在 Decoder-only 架构中，由于因果掩码（Causal Mask）和 Padding 本身就打破了排列不变性，大模型仅靠 Mask 就能隐式学到部分位置和顺序信息。


17. **ALiBi (Attention with Linear Biases, 线性注意力偏置)**
* **来源**：*Train Short, Test Long* (2021)，广泛用于 Bloom 模型。
* **原理**：完全不向 Embedding 注入任何位置信息。而是在 Attention 计算时，直接对每一个 Head 的 $Q K^T$ 矩阵赋予一个随相对距离线性递减的惩罚项（$-m \cdot |i-j|$），其中 $m$ 是每个 Head 固定的超参数斜率。外推性能极强。


18. **Sandwich Positioning (三明治位置编码)**
* **来源**：CogView / 改进视觉与文本多模态对齐。
* **原理**：在 Transformer 的各个 Layer 之间交叉重复注入位置编码（Layer 输入加一次，Layer 内部再加一次），以防位置信息在深层网络中被稀释。



---

### 六、 2D / 3D / 多维位置编码 (Multidimensional PEs)

针对计算机视觉（ViT）、视频模型以及 3D 点云/科学计算的多维空间编码。

19. **2D Sinusoidal / Learned PE (2D 空间位置编码)**
* **来源**：ViT (Vision Transformer) / DETR。
* **原理**：将高度（X 轴）和宽度（Y 轴）分别计算一维的位置编码（各占 $d/2$ 维度），然后拼接（Concat）成一个完整的 2D 位置编码加到图像 Patch 嵌入上。


20. **3D Grid PE (3D 网格位置编码)**
* **来源**：视频 Transformer（如 Timesformer）或 3D 医疗图像、点云处理。
* **原理**：在 2D 的基础上引入时间轴（Time 轴）或深度轴（Z 轴），将 X, Y, T 三个维度的编码进行组合或加和。


21. **CP-Vis (Conditional Positional Encoding / CPE)**
* **来源**：*Conditional Positional Encodings for Vision Transformers* (2021)。
* **原理**：利用深度可分离卷积（Depth-wise Convolution）动态地根据输入的图像内容生成位置编码。它天然支持任意分辨率输入（因为卷积具有平移不变性和对尺度的自适应性）。


22. **RoPE-2D / Axial RoPE (轴向旋转位置编码)**
* **来源**：多模态大模型（如图像/视频大模型 Qwen-VL, CogVLM）。
* **原理**：将 RoPE 扩展到二维，对图像的行坐标和列坐标分别应用不同角度的旋转矩阵。

# 七、旋转位置编码 RoPE：从原理到代码

> 本节从一维文本 RoPE 开始，逐步推导到二维空间位置编码，并最终对应到车辆规划项目 `agent_policy` 使用的 **LieRE2D**。建议按顺序运行后面的代码单元。

## 1. 为什么要旋转 Q 和 K，而不是把位置向量加到 token 上？

标准自注意力在忽略缩放时计算

$$s_{ij}=q_i^T k_j.$$

若直接使用绝对位置加法 $x_i+p_i$，位置信息会混入内容表示，而且 $s_{ij}$ 不会天然只依赖相对位置 $i-j$。RoPE 的思路是：先由位置 $i$ 构造一个正交旋转矩阵 $R_i$，只旋转 Query 和 Key：

$$\tilde q_i=R_iq_i,\qquad \tilde k_j=R_jk_j.$$

于是注意力内积变为

$$\tilde q_i^T\tilde k_j=q_i^TR_i^TR_jk_j.$$

如果旋转满足 $R_i^TR_j=R_{j-i}$，注意力分数就只通过 $j-i$ 感知相对位置。由于 $R_i$ 是正交矩阵，还会保持向量范数：$\|R_iq_i\|_2=\|q_i\|_2$。因此 RoPE 不改变 Q/K 的尺度，只改变不同位置向量之间的夹角。

### 与其他位置编码的区别

| 方法 | 注入位置 | 是否天然相对 | 是否改变 token/QK 范数 | 推理长度外推 |
|---|---|---:|---:|---:|
| 绝对正弦编码 | 加到 token | 否 | 会改变 token | 一般 |
| Learned PE | 加到 token | 否 | 会改变 token | 受训练最大长度限制 |
| Relative Bias / ALiBi | Attention logits | 是 | 不改变 Q/K | 较好 |
| RoPE | 旋转 Q/K | 是 | 不改变 | 需配合缩放策略 |
| LieRE | 由连续坐标生成旋转并作用 Q/K | 取决于生成元结构；2D 小块时可精确相对 | 不改变 | 适合连续几何坐标 |


## 2. 一维 RoPE 的数学推导

### 2.1 二维子空间中的旋转

对 head_dim 中相邻的两个通道 $(x_{2m},x_{2m+1})$，位置 $p$ 对应的旋转为

$$R_m(p)=\begin{bmatrix}\cos(p\theta_m)&-\sin(p\theta_m)\\\sin(p\theta_m)&\cos(p\theta_m)\end{bmatrix},$$

其中 $\theta_m=\text{base}^{-2m/d_h}$，$d_h$ 是每个注意力头的维度。低维通道旋转较快，负责区分近距离；高维通道旋转较慢，负责表达长距离关系。

将整个 $d_h$ 维向量分成 $d_h/2$ 个二维块，完整旋转矩阵就是这些 $R_m(p)$ 的块对角矩阵。因此 **head_dim 必须是偶数**。

### 2.2 相对位置性质

二维旋转满足 $R(a)^TR(b)=R(b-a)$。于是每个频率块都有

$$[R_m(i)q_i]^T[R_m(j)k_j]=q_i^TR_m(j-i)k_j.$$

这就是“使用绝对位置构造旋转，却在注意力内积中得到相对位置”的关键。注意，RoPE 并不是向 attention score 直接加入距离偏置；它让内容向量与相对位置发生乘性交互。

### 2.3 复数视角

把一对实数通道视作复数 $z=x_{2m}+ix_{2m+1}$，旋转等价于

$$z_p=z\,e^{ip\theta_m}.$$

而 Query 与 Key 的复内积中相位相减，得到 $e^{i(j-i)\theta_m}$。代码通常不用复数，而是使用 `cos/sin` 和成对通道运算，以便兼容混合精度和 ONNX。


In [ ]:
import math
import torch

torch.manual_seed(7)

def build_rope_cache(seq_len: int, head_dim: int, base: float = 10_000.0,
                     device=None, dtype=torch.float32):
    """构造一维 RoPE 的 cos/sin 表。

    Returns:
        cos, sin: [seq_len, head_dim // 2]
    """
    if head_dim % 2 != 0:
        raise ValueError(f"head_dim 必须为偶数，当前为 {head_dim}")
    # m = 0,...,head_dim/2-1；inv_freq: [head_dim/2]
    inv_freq = base ** (-torch.arange(0, head_dim, 2, device=device).float() / head_dim)
    # positions: [seq_len]；outer 后 angles: [seq_len, head_dim/2]
    positions = torch.arange(seq_len, device=device, dtype=torch.float32)
    angles = torch.outer(positions, inv_freq)
    return angles.cos().to(dtype), angles.sin().to(dtype)

def apply_rope_1d(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    """对 x=[B,H,N,Dh] 施加一维 RoPE，返回形状不变。"""
    B, H, N, Dh = x.shape
    if Dh % 2 != 0:
        raise ValueError("每个 head 的维度必须为偶数")
    # 相邻通道组成二维旋转块：[B,H,N,Dh] -> [B,H,N,Dh/2]
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    # [N,Dh/2] -> [1,1,N,Dh/2]，沿 batch 和 head 广播。
    cos = cos[:N].unsqueeze(0).unsqueeze(0)
    sin = sin[:N].unsqueeze(0).unsqueeze(0)
    y_even = x_even * cos - x_odd * sin
    y_odd = x_even * sin + x_odd * cos
    # stack: [B,H,N,Dh/2,2]；flatten 恢复 [B,H,N,Dh]。
    return torch.stack([y_even, y_odd], dim=-1).flatten(-2)

B, H, N, Dh = 2, 4, 6, 8
q = torch.randn(B, H, N, Dh)
k = torch.randn(B, H, N, Dh)
cos, sin = build_rope_cache(N, Dh)
q_rot = apply_rope_1d(q, cos, sin)
k_rot = apply_rope_1d(k, cos, sin)

print('q shape:', q.shape, 'q_rot shape:', q_rot.shape)
print('最大范数误差:', (q.norm(dim=-1) - q_rot.norm(dim=-1)).abs().max().item())

## 3. 用实验验证“只依赖相对位置”

验证时要避免一个常见误区：不能直接比较不同随机 $q_i,k_j$ 的分数，因为内容本身已经不同。正确做法是固定同一对内容向量，只改变它们放置的绝对位置；只要位置差相同，旋转后的内积就应相同。


In [ ]:
def rotate_single_position(x: torch.Tensor, position: int, head_dim: int):
    """x=[Dh] -> [Dh]，使用指定绝对位置旋转。"""
    cos, sin = build_rope_cache(position + 1, head_dim)
    return apply_rope_1d(x.view(1, 1, 1, head_dim),
                         cos[position:position + 1],
                         sin[position:position + 1]).view(head_dim)

q_content = torch.randn(Dh)
k_content = torch.randn(Dh)

# 两对绝对位置分别是 (2, 5) 和 (11, 14)，相对距离都为 3。
score_a = torch.dot(rotate_single_position(q_content, 2, Dh),
                    rotate_single_position(k_content, 5, Dh))
score_b = torch.dot(rotate_single_position(q_content, 11, Dh),
                    rotate_single_position(k_content, 14, Dh))
print('score_a:', score_a.item())
print('score_b:', score_b.item())
print('相同相对距离下的误差:', (score_a - score_b).abs().item())
assert torch.allclose(score_a, score_b, atol=1e-5)

## 4. RoPE 在多头注意力中的正确位置

标准处理顺序是：

1. 输入 token：`x [B,N,D_model]`；
2. 线性投影得到 `qkv [B,N,3*H*Dh]`；
3. reshape 为 `q/k/v [B,N,H,Dh]`；
4. 可选 QK-Norm；
5. **只对 q 和 k 应用 RoPE，不旋转 v**；
6. transpose 为 `[B,H,N,Dh]`；
7. 计算 $QK^T/\sqrt{d_h}$，得到 `[B,H,N,N]`；
8. softmax 后乘 V，再合并多头。

为什么不旋转 V？位置需要影响“从哪里读取”的匹配关系，而 V 承载被读取的内容。若旋转 V，输出内容还会残留绝对位置相关的旋转坐标系，破坏 RoPE 简洁的相对位置解释。


In [ ]:
def rope_attention(q_bnhd, k_bnhd, v_bnhd, padding_mask=None):
    """教学版 RoPE attention。输入 q/k/v 均为 [B,N,H,Dh]。"""
    B, N, H, Dh = q_bnhd.shape
    cos, sin = build_rope_cache(N, Dh, device=q_bnhd.device, dtype=q_bnhd.dtype)
    # apply_rope_1d 需要 [B,H,N,Dh]，所以先交换 token/head 维。
    q = apply_rope_1d(q_bnhd.transpose(1, 2), cos, sin)
    k = apply_rope_1d(k_bnhd.transpose(1, 2), cos, sin)
    v = v_bnhd.transpose(1, 2)
    scores = q @ k.transpose(-2, -1) / math.sqrt(Dh)  # [B,H,N,N]
    if padding_mask is not None:
        # padding_mask [B,N]，True 表示无效 key；扩成 [B,1,1,N]。
        scores = scores.masked_fill(padding_mask[:, None, None, :], -1e4)
    weights = scores.softmax(dim=-1)
    out = weights @ v                              # [B,H,N,Dh]
    return out.transpose(1, 2).reshape(B, N, H * Dh)  # [B,N,H*Dh]

q_demo = torch.randn(B, N, H, Dh)
k_demo = torch.randn(B, N, H, Dh)
v_demo = torch.randn(B, N, H, Dh)
padding = torch.zeros(B, N, dtype=torch.bool)
padding[:, -1] = True
print('attention output:', rope_attention(q_demo, k_demo, v_demo, padding).shape)

# 八、从一维 RoPE 到二维空间旋转编码

车辆场景 token 没有天然的一维序号含义。邻车、实线段、虚线段、中心线段和历史规划 token 的关键位置变量是物理坐标 $(x,y)$。如果仅按 token 拼接顺序做一维 RoPE，会让“数组中相邻”错误地等价于“物理空间相邻”。

## 1. Axial RoPE（轴向 RoPE）

最直接的二维扩展是把 head_dim 分为两部分：一部分使用 x 坐标旋转，另一部分使用 y 坐标旋转。它简单、计算快，但 x/y 完全解耦，而且人为规定了各轴占用的通道。

## 2. LieRE（Lie-group Relative Encoding）

更一般的做法是为每个坐标轴学习一个**斜对称生成元**。二维坐标 $p=(x,y)$ 对应

$$A(p)=xA_x+yA_y,\qquad A_x^T=-A_x,\;A_y^T=-A_y,$$

再通过矩阵指数映射到旋转群：

$$R(p)=\exp(A(p)).$$

斜对称矩阵的指数一定是正交矩阵，因此 $R(p)^TR(p)=I$，Q/K 的范数仍被保持。与固定频率 RoPE 不同，生成元可以训练，并且可以按 Transformer 层、按注意力头分别学习。

### 一个重要的严格性说明

要严格满足 $R(p_i)^TR(p_j)=R(p_j-p_i)$，不同坐标轴生成元需要可交换：$A_xA_y=A_yA_x$。当前 `agent_policy` 默认 `generator_dim=2`，每个生成元块都是二维斜对称矩阵，二维斜对称空间只有一个自由度，所以这些块天然可交换，能保留标准 RoPE 的相对位置性质。

若把 `generator_dim` 改为大于 2，一般的斜对称矩阵未必可交换，此时 LieRE 仍能提供正交、可学习的几何旋转，但不能不加条件地宣称它只依赖坐标差。理解这一点对于调参和解释模型非常重要。


In [ ]:
class LieRE2DTutorial(torch.nn.Module):
    """与 agent_policy LieRE2D 同构的教学实现。"""

    def __init__(self, head_dim, num_heads, depth, generator_dim=2,
                 rotary_per_layer=True, rotary_per_head=True):
        super().__init__()
        if head_dim % generator_dim != 0:
            raise ValueError('head_dim 必须能被 generator_dim 整除')
        self.head_dim = head_dim
        self.num_heads = num_heads
        self.depth = depth
        self.generator_dim = generator_dim
        self.num_generators = head_dim // generator_dim
        self.rotary_per_layer = rotary_per_layer
        self.rotary_per_head = rotary_per_head

        Lg = depth if rotary_per_layer else 1
        Rg = self.num_generators * num_heads if rotary_per_head else self.num_generators
        # 两个坐标轴 × 层 × 旋转块 × G × G。raw 参数无需自身满足斜对称。
        raw = torch.rand(2, Lg, Rg, generator_dim, generator_dim) * (2 * math.pi)
        self.generator_raw_params = torch.nn.Parameter(raw)

    def forward(self, positions, layer_idx=0, dtype=None):
        """positions [B,N,2] -> rotations [B,N,Rg,G,G]。"""
        dtype = positions.dtype if dtype is None else dtype
        layer = layer_idx if self.rotary_per_layer else 0
        generators = self.generator_raw_params[:, layer]  # [2,Rg,G,G]
        # 任意矩阵 M 取严格上三角 U，再构造 U-U^T，保证 bases^T=-bases。
        upper = torch.triu(generators, diagonal=1)
        bases = upper - upper.transpose(-1, -2)          # [2,Rg,G,G]
        # bnd,drgh->bnrgh：对 d=2 个坐标轴做加权和。
        # [B,N,2] × [2,Rg,G,G] -> [B,N,Rg,G,G]。
        algebra_element = torch.einsum('bnd,drgh->bnrgh',
                                       positions.float(), bases.float())
        # 对每个 G×G 斜对称矩阵做 exp，得到正交旋转矩阵。
        return torch.matrix_exp(algebra_element).to(dtype)

    def apply_to_qk(self, q, k, rotations):
        """q/k [B,N,H,Dh] -> 同形状旋转结果。"""
        B, N, H, Dh = q.shape
        G = self.generator_dim
        R = self.num_generators
        # 每个 head 拆为 R 个 G 维旋转块。
        q_blocks = q.reshape(B, N, H, R, G)             # [B,N,H,R,G]
        k_blocks = k.reshape(B, N, H, R, G)
        if self.rotary_per_head:
            rot = rotations.reshape(B, N, H, R, G, G)  # [B,N,H,R,G,G]
        else:
            rot = rotations.unsqueeze(2)                # [B,N,1,R,G,G]
        # 行向量块乘旋转矩阵；head=1 时会自动广播到 H。
        q_rot = torch.einsum('bnhgj,bnhgij->bnhgi', q_blocks, rot)
        k_rot = torch.einsum('bnhgj,bnhgij->bnhgi', k_blocks, rot)
        return q_rot.reshape(B, N, H, Dh), k_rot.reshape(B, N, H, Dh)

liere = LieRE2DTutorial(head_dim=8, num_heads=4, depth=3, generator_dim=2)
positions = torch.randn(2, 10, 2)  # [B=2,N=10,(x,y)]
q = torch.randn(2, 10, 4, 8)       # [B,N,H,Dh]
k = torch.randn_like(q)
rotations = liere(positions, layer_idx=1)
q_rot, k_rot = liere.apply_to_qk(q, k, rotations)
print('rotations:', rotations.shape)
print('q_rot:', q_rot.shape, 'k_rot:', k_rot.shape)

## 3. LieRE 的数值性质与单元测试

实现旋转位置编码后，至少应测试：

1. 输出 shape 与输入一致；
2. $R^TR\approx I$；
3. 旋转前后 Q/K 范数基本一致；
4. 梯度能够回传到生成元参数；
5. 当 `generator_dim=2` 时，共同平移所有 token 不应改变固定内容对的注意力内积；
6. float16/bfloat16 下最好仍用 float32 构造矩阵指数，再转换回模型 dtype。


In [ ]:
# 1) 正交性：R^T R 应接近单位阵。
G = liere.generator_dim
identity = torch.eye(G)
orthogonal_error = (rotations.transpose(-1, -2) @ rotations - identity).abs().max()
print('最大正交误差:', orthogonal_error.item())

# 2) 范数保持。
norm_error = (q.norm(dim=-1) - q_rot.norm(dim=-1)).abs().max()
print('最大范数误差:', norm_error.item())

# 3) 梯度检查：必须比较不同位置的 Q/K。若比较同一位置并施加同一个
# 正交旋转，内积会严格保持，对旋转参数的梯度理论上接近 0。
loss = (q_rot[:, 1:] * k_rot[:, :-1]).mean()
loss.backward()
grad = liere.generator_raw_params.grad
print('生成元梯度 shape:', grad.shape, '梯度绝对值均值:', grad.abs().mean().item())

assert orthogonal_error < 1e-4
assert norm_error < 1e-4
assert grad is not None and torch.isfinite(grad).all()
assert grad.abs().max() > 1e-8

# 九、`agent_policy` 中 LieRE2D 的真实数据流

项目实现位于：

- `agent_policy/src/models/components/encoder/liere.py`：生成旋转矩阵并旋转 Q/K；
- `encoder/lane_split.py::TokenCoordinateGenerator`：给不同类型 token 生成统一二维物理坐标；
- `encoder/encoder.py::PrefusionBlock`：场景 token 自注意力中的 LieRE；
- `decoder/decoder.py`：轨迹 token 与场景 token 注意力中的 LieRE；
- `decoder/global_attention.py`：全局注意力中的 LieRE；
- `models/components/fmslm.py` 和 `fmslm_accel.py`：根据配置实例化共享的 `LieRE2D`。

## 1. 项目中的核心 shape

设 $B$ 为 batch，$N$ 为本层 token 总数，$H$ 为注意力头数，$D_h$ 为 head_dim，$G$ 为 generator_dim，$R=D_h/G$ 为每头的旋转块数：

| 张量 | 形状 | 含义 |
|---|---|---|
| `coords_2d` | `[B,N,2]` | 每个 token 的物理 `(x,y)` 坐标 |
| `q/k` | `[B,N,H,Dh]` | QKNorm 后、attention transpose 前的 Q/K |
| raw generators | `[2,Lg,H*R,G,G]` | 两坐标轴、层、头内旋转块的可学习原始参数 |
| skew bases | `[2,H*R,G,G]` | 当前层的斜对称生成元 |
| `generator_pos` | `[B,N,H*R,G,G]` | 坐标加权后的 Lie 代数元素 |
| `rotations` | `[B,N,H*R,G,G]` | `matrix_exp` 后的正交矩阵 |
| blocked q/k | `[B,N,H,R,G]` | 把每个 head 拆成旋转小块 |
| rotated q/k | `[B,N,H,Dh]` | 恢复原 head_dim 后的结果 |

默认 `generator_dim=2` 时，`num_generators=head_dim//2`。若 `head_dim=24`、`num_heads=8`，则每个 token 有 `8*12=96` 个 $2\times2$ 旋转块。

## 2. 坐标如何对齐 token

LieRE 最危险的 bug 通常不是矩阵公式，而是 **token 顺序与坐标顺序错位**。项目必须保证 token 拼接和 `coords_2d` 拼接使用完全相同的顺序，例如：全局条件 token、自车 token、邻车 token、各类车道段 token、历史规划 token。无空间含义的全局 token 通常使用 `(0,0)`；无效 token 的坐标值本身不重要，但必须同时被 `key_padding_mask` 屏蔽。

Decoder 中轨迹 token 的坐标可能来自当前噪声状态的动态预测。因此不同扩散/流匹配时刻下，LieRE 旋转也会随轨迹空间位置变化；这与文本中固定 token index 的 RoPE 有本质区别。


## 3. 配置参数如何理解

典型配置结构：

```yaml
liere:
  enable: true
  generator_dim: 2
  rotary_per_layer: true
  rotary_per_head: true
```

- `enable`：关闭时不构造/应用旋转，适合消融实验。
- `generator_dim=2`：推荐默认值。计算便宜、严格正交，并具有清晰的二维旋转解释。必须整除 `head_dim`。
- `rotary_per_layer=true`：每层学习独立生成元，表达力更强，但参数量和矩阵指数计算量增加。
- `rotary_per_head=true`：每个 head 学习独立的空间频率/方向，允许不同 head 关注不同尺度。关闭时所有 head 共享旋转块。

参数量近似为

$$2\times L_g\times(H\cdot D_h/G)\times G^2,$$

其中第一项 2 对应 x/y 两个坐标轴。默认 $G=2$ 时参数量通常不大，但 `torch.matrix_exp` 会对每个 batch、token、旋转块执行，因此运行开销比查表式 RoPE 高。


# 十、工程实践、常见错误与调试清单

## 1. 常见错误

1. **旋转了 V**：标准 RoPE/LieRE 只旋转 Q/K。
2. **Q/K layout 错误**：项目 LieRE 接受 `[B,N,H,Dh]`，而 PyTorch SDPA 常用 `[B,H,N,Dh]`。transpose 的位置必须明确。
3. **head_dim 不能整除 generator_dim**：reshape 会失败，应在构造函数中提前断言。
4. **坐标单位过大**：物理坐标若达到数千米，生成的旋转角可能变化过快。可考虑以自车为原点、使用米制局部坐标或对坐标缩放。
5. **全局坐标原点漂移**：若每个场景坐标系定义不一致，会让模型学习无意义的绝对相位。车辆规划通常应使用 ego-centric 局部坐标。
6. **padding token 未屏蔽**：即使它的坐标设为 0，也会作为有效 Key 参与 softmax。坐标和 mask 是两套机制。
7. **在 float16 直接 `matrix_exp`**：可能产生明显数值误差。当前项目先把坐标和生成元计算提升到 float32，再转回目标 dtype。
8. **误以为任何高维 LieRE 都严格相对**：生成元不交换时，相对位置等式一般不成立。
9. **缓存可学习 LieRE 旋转**：训练中生成元会更新，跨 optimizer step 缓存会变成陈旧结果；只能缓存固定 RoPE 的 cos/sin。
10. **不同模块重复实例化**：若设计目标是 encoder/decoder 共享空间频率，应传递同一个 `LieRE2D` 实例，而不是各自 new 一个。

## 2. 推荐调试顺序

- 打印 token 数 N，并断言 `coords_2d.shape[:2] == tokens.shape[:2]`；
- 断言 `q.shape == k.shape == [B,N,H,Dh]`；
- 检查 `R^T R-I` 最大误差；
- 检查旋转前后 Q/K 范数误差；
- 检查 padding mask 中 True/False 语义；
- 检查所有坐标是否有限：`torch.isfinite(coords).all()`；
- 观察生成元梯度是否为 0、NaN 或异常大；
- 做 `enable_liere=false/true` 消融，比较验证集规划误差、碰撞率和训练吞吐；
- 可视化不同 head 的旋转角随 `(x,y)` 的变化，判断是否学到不同空间尺度。


In [ ]:
def diagnose_liere(module, positions, q, k, layer_idx=0):
    """一个可直接用于训练前冒烟检查的诊断函数。"""
    assert positions.ndim == 3 and positions.shape[-1] == 2
    assert q.shape == k.shape and q.ndim == 4
    assert positions.shape[:2] == q.shape[:2]
    assert torch.isfinite(positions).all()
    rotations = module(positions, layer_idx=layer_idx, dtype=q.dtype)
    q_rot, k_rot = module.apply_to_qk(q, k, rotations)
    G = rotations.shape[-1]
    eye = torch.eye(G, device=rotations.device, dtype=rotations.dtype)
    report = {
        'positions_shape': tuple(positions.shape),
        'rotations_shape': tuple(rotations.shape),
        'q_shape': tuple(q.shape),
        'orthogonal_max_error': float((rotations.transpose(-1, -2) @ rotations - eye).abs().max()),
        'q_norm_max_error': float((q.norm(dim=-1) - q_rot.norm(dim=-1)).abs().max()),
        'k_norm_max_error': float((k.norm(dim=-1) - k_rot.norm(dim=-1)).abs().max()),
    }
    return report

# 前面的教学模块可以直接接受该诊断。
print(diagnose_liere(liere, positions, q.detach(), k.detach(), layer_idx=1))

# 十一、进一步学习与练习

## 推荐阅读

1. Su et al., **RoFormer: Enhanced Transformer with Rotary Position Embedding**。
2. `agent_policy/src/models/components/encoder/liere.py`，结合本节 shape 表逐行阅读。
3. Lie 群与 Lie 代数基础：重点理解斜对称矩阵、矩阵指数、正交群 $SO(n)$。
4. Position Interpolation、NTK-aware RoPE、YaRN：用于理解文本长上下文中的 RoPE 外推问题。

## 建议练习

1. 实现 Axial RoPE，并比较它与 LieRE2D 的参数量和运行时间。
2. 固定 token 内容，对所有二维坐标加相同平移，验证 `generator_dim=2` 时注意力分数是否保持。
3. 把 `generator_dim` 改为 4，计算 $\|[A_x,A_y]\|$，观察非交换生成元如何破坏严格相对性。
4. 在 `agent_policy` 中分别设置 `rotary_per_head=false/true` 做消融。
5. 统计真实场景 `coords_2d` 的范围，测试不同坐标缩放系数对旋转角和训练稳定性的影响。
6. 给 `LieRE2D` 增加构造参数断言、正交性测试、混合精度测试和 ONNX 导出测试。

完成这些练习后，应能够从数学、张量实现和车辆规划工程三个层面解释旋转位置编码，而不只是记住“给 Q/K 乘一个旋转矩阵”。
